# 00. 베이스라인 + 공통 교차검증

- 담당: **① 장성은 (팀장)**
- 팀 공통 기준 점수(CV Macro F1)예요. 모든 실험은 이 점수와 비교해요.
- 피처: 팀 기본 피처 `binarize_mutations` (WT·결측 = 0, 변이 = 1)
- 모델: XGBoost (운영진 베이스라인과 비슷한 설정)
- 누수 방지: 전처리와 모델을 Pipeline으로 묶어 fold의 학습 부분에만 fit. test는 형식 확인과 최종 추론에만.

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

In [ ]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
import xgboost as xgb
print('xgboost', xgb.__version__)

## 1. 데이터

In [ ]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

## 2. 모델 (Pipeline)

In [ ]:
XGB_PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6,
                  tree_method="hist", random_state=SEED, n_jobs=-1)

def make_pipeline():
    return Pipeline([
        ("features", basic_transformer()),
        ("model", xgb.XGBClassifier(**XGB_PARAMS)),
    ])

## 3. 공통 교차검증 (5-fold, 로컬 기준 약 10분)

In [ ]:
cv = run_cv(make_pipeline(), X, y, folds=folds)

## 4. 클래스별 F1
Macro F1은 26개 클래스를 같은 비중으로 평균해요. 낮은 클래스가 개선 포인트예요.

In [ ]:
per_class_f1(y, cv['oof'], le.classes_).round(3)

## 5. 기록

In [ ]:
log_result(owner="J_leader", notebook="00_baseline.ipynb", experiment="baseline_xgb",
           features="basic(WT=0,변이=1)", model="XGBoost", params=XGB_PARAMS, cv=cv,
           notes="운영진 베이스라인 설정 + 공통 CV")

## 6. 전체 train 학습 → test 예측 → 제출 파일
제출은 팀장이 하루 4회 안에서 해요.

In [ ]:
final_model = make_pipeline().fit(X, y)
pred = le.inverse_transform(final_model.predict(test.drop(columns=["ID"])))
sub = make_submission(test, pred, "00_baseline_xgb.csv")
sub.head()